In [43]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
import requests
from dotenv import load_dotenv
load_dotenv()

@tool
def get_weather(city: str) -> str:
    """Get weather of a city"""
    api_key = "3175855318014ea3ae697f91055857b9"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    data = requests.get(url).json()
    print(f"{city}: {data['main']['temp']}°C, {data['weather'][0]['description']}")
    return f"{city}: {data['main']['temp']}°C, {data['weather'][0]['description']}"

@tool              
def get_username():
    """Get the name of the user"""
    return f"My name is bob"
    
agent = create_agent(
    model=ChatOpenAI(model="gpt-5.4-nano"),   # try bigger than 0.6b
    tools=[get_weather, get_username],
    system_prompt="You are a helpful assistant. Invoke necessary tools as needed and always answer all the questions using the username. call necessary tool to get the username for example. Hi Username",
)


result = agent.invoke({
    "messages": [
        {"role": "user", "content": "How is the weather in Chicago ?"}
    ]
})



Chicago: 6.86°C, broken clouds


In [44]:
for msg in result["messages"]:
    if msg.__class__.__name__ == "HumanMessage":
        print(f"\n👤 User: {msg.content}")

    elif msg.__class__.__name__ == "AIMessage":
        if msg.tool_calls:
            print(f"\n🤖 AI (tool call): {msg.tool_calls}")
        else:
            print(f"\n🤖 AI: {msg.content}")

    elif msg.__class__.__name__ == "ToolMessage":
        print(f"\n🛠️ Tool ({msg.name}): {msg.content}")


👤 User: How is the weather in Chicago ?

🤖 AI (tool call): [{'name': 'get_username', 'args': {}, 'id': 'call_4kDJjKBz8LmOzKg7rb4RVAN5', 'type': 'tool_call'}]

🛠️ Tool (get_username): My name is bob

🤖 AI (tool call): [{'name': 'get_weather', 'args': {'city': 'Chicago'}, 'id': 'call_KZGijRyb4Cy8Ie8vWfnOVRmv', 'type': 'tool_call'}]

🛠️ Tool (get_weather): Chicago: 6.86°C, broken clouds

🤖 AI: Hi bob — the weather in Chicago is **6.86°C** with **broken clouds**.
